<div dir="rtl">

# ۰۴ — احتمال، احتمال شرطی، استقلال و قضیه بیز

**فاز دوم:** ترجمه داده‌های واقعی به زبان احتمالات

این نوت‌بوک گام اول از فاز دوم است و تسک‌های زیر را پوشش می‌دهد:
1. تعریف رویدادها (تسک ۱)
2. احتمال‌های حاشیه‌ای و مشترک (تسک ۲)
3. احتمال شرطی و زیرگروه‌ها (تسک ۳ و الف.۱)
4. بررسی مفهوم استقلال (تسک ۴)
5. قضیه بیز (تسک ۹)
6. نقش وزن‌های نمونه‌گیری در احتمال تجربی (تسک الف.۴)

</div>

In [8]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np

sys.path.append(str(Path("..").resolve() / "src"))
import nhanes as nh

pd.set_option("display.width", 160)
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")

TAB_DIR = Path("..") / "reports" / "phase2"
TAB_DIR.mkdir(parents=True, exist_ok=True)

# Load analytic sample from phase 1
DATA_PATH = Path("..") / "data" / "processed" / "analytic_sample.csv"
df = pd.read_csv(DATA_PATH)
print(f"Analytic sample loaded: {len(df):,} adults")

Analytic sample loaded: 7,996 adults


<div dir="rtl">

## ۱. تعریف رویدادها (تسک ۱)

ابتدا دو رویداد مرتبط با سوال پروژه را تعریف می‌کنیم:
- **رویداد $A$ (خواب کوتاه):** فرد به طور میانگین کمتر از ۷ ساعت در هفته می‌خوابد (`short_sleep == 1`).
- **رویداد $B$ (سیگاری فعلی):** وضعیت مصرف سیگار فرد "فعلی" است (`smoking_status == 'current'`).

این دو رویداد پایه محاسبات احتمالی ما در این نوت‌بوک خواهند بود.

</div>

In [9]:
event_A = df['short_sleep'] == 1
event_B = df['smoking_status'] == 'current'

# Total number of observations
N = len(df)

<div dir="rtl">

## ۲. احتمال‌های حاشیه‌ای و مشترک (تسک ۲)

احتمال تجربی هر رویداد برابر است با فراوانی وقوع آن رویداد تقسیم بر حجم نمونه تحلیلی.

</div>

In [10]:
p_A = event_A.mean()
p_B = event_B.mean()
p_A_and_B = (event_A & event_B).mean()

print(f"P(A) - Probability of short sleep: {p_A:.4f}")
print(f"P(B) - Probability of current smoker: {p_B:.4f}")
print(f"P(A ∩ B) - Joint probability of short sleep AND current smoker: {p_A_and_B:.4f}")

P(A) - Probability of short sleep: 0.1995
P(B) - Probability of current smoker: 0.1442
P(A ∩ B) - Joint probability of short sleep AND current smoker: 0.0406


<div dir="rtl">

## ۳. احتمال شرطی و زیرگروه‌ها (تسک ۳ و الف.۱)

احتمال شرطی نشان می‌دهد که با دانستن وقوع یک رویداد، احتمال رویداد دیگر چقدر تغییر می‌کند.
$P(A|B) = \frac{P(A \cap B)}{P(B)}$

در اینجا احتمال خواب کوتاه ($A$) را به شرط سیگاری بودن ($B$) و همچنین در زیرگروه‌های جنسیتی بررسی می‌کنیم.

</div>

In [11]:
# Conditional probability P(A|B)
p_A_given_B = p_A_and_B / p_B
print(f"P(A|B) - Probability of short sleep GIVEN current smoker: {p_A_given_B:.4f}")

# Conditional probability given NEVER smoker (for comparison)
event_B_never = df['smoking_status'] == 'never'
p_A_given_never_smoker = (event_A & event_B_never).mean() / event_B_never.mean()
print(f"P(A|Never Smoker) - Probability of short sleep GIVEN never smoker: {p_A_given_never_smoker:.4f}")

print("\n--- Task A.1: Conditional Probabilities in Subgroups ---")
# Subgroups based on Gender (1 = Male, 2 = Female)
male_mask = df['RIAGENDR'] == 1
female_mask = df['RIAGENDR'] == 2

p_A_given_male = (event_A & male_mask).mean() / male_mask.mean()
p_A_given_female = (event_A & female_mask).mean() / female_mask.mean()

print(f"P(A|Male) - Probability of short sleep GIVEN male: {p_A_given_male:.4f}")
print(f"P(A|Female) - Probability of short sleep GIVEN female: {p_A_given_female:.4f}")

P(A|B) - Probability of short sleep GIVEN current smoker: 0.2819
P(A|Never Smoker) - Probability of short sleep GIVEN never smoker: 0.1791

--- Task A.1: Conditional Probabilities in Subgroups ---
P(A|Male) - Probability of short sleep GIVEN male: 0.2172
P(A|Female) - Probability of short sleep GIVEN female: 0.1851


<div dir="rtl">

**تفسیر:** شرط کردن روی ویژگی «سیگاری فعلی»، احتمال رویداد «خواب کوتاه» را از احتمال حاشیه‌ای آن ($0.1995$) به صورت محسوسی به بالا ($0.2819$) تغییر می‌دهد. همچنین مردان احتمال خواب کوتاه بیشتری نسبت به زنان دارند.

</div>

<div dir="rtl">

## ۴. بررسی مفهوم استقلال (تسک ۴)

دو رویداد مستقل هستند اگر $P(A \cap B) = P(A) \times P(B)$.

</div>

In [12]:
p_A_times_p_B = p_A * p_B
difference = abs(p_A_and_B - p_A_times_p_B)

print(f"P(A ∩ B) = {p_A_and_B:.4f}")
print(f"P(A) * P(B) = {p_A_times_p_B:.4f}")
print(f"Difference = {difference:.4f}")

if difference < 0.01:
    print("Conclusion: The events appear to be independent.")
else:
    print("Conclusion: The events DO NOT appear to be independent.")

P(A ∩ B) = 0.0406
P(A) * P(B) = 0.0288
Difference = 0.0119
Conclusion: The events DO NOT appear to be independent.


<div dir="rtl">

**تفسیر استقلال:** از آنجا که حاصل‌ضرب احتمالات حاشیه‌ای با احتمال مشترک آن‌ها یکسان نیست (و تفاوت قابل توجهی دارد)، این داده‌ها با فرض استقلال سازگار نیستند. وابستگی آماری بین وضعیت مصرف سیگار و مدت خواب وجود دارد.

</div>

<div dir="rtl">

## ۵. قضیه بیز (تسک ۹)

ما احتمال خواب کوتاه به شرط سیگاری بودن را حساب کردیم ($P(A|B)$). حالا با استفاده از قضیه بیز، این احتمال را به روز رسانی می‌کنیم: اگر بدانیم شخصی خواب کوتاه دارد، با چه احتمالی سیگاری فعلی است ($P(B|A)$)؟

$P(B|A) = \frac{P(A|B) \times P(B)}{P(A)}$

</div>

In [13]:
# Calculate P(B|A) using Bayes' Theorem
p_B_given_A_bayes = (p_A_given_B * p_B) / p_A

# Verification: Calculate P(B|A) directly from empirical frequencies
p_B_given_A_empirical = p_A_and_B / p_A

print(f"P(B|A) calculated via Bayes' Theorem: {p_B_given_A_bayes:.4f}")
print(f"P(B|A) calculated directly (empirical): {p_B_given_A_empirical:.4f}")

P(B|A) calculated via Bayes' Theorem: 0.2038
P(B|A) calculated directly (empirical): 0.2038


<div dir="rtl">

## ۶. نقش وزن‌های نمونه‌گیری در احتمال تجربی (تسک الف.۴)

احتمال تجربی خام فقط فراوانی در نمونه را نشان می‌دهد. برای تعمیم این احتمال به کل جامعه آمریکا، باید وزن‌های پیمایش (`WTINT2YR`) را اعمال کنیم.

</div>

In [14]:
W = df['WTINT2YR']
p_A_weighted = nh.weighted_share(event_A, W)
p_B_weighted = nh.weighted_share(event_B, W)

print("--- Comparison of Unweighted vs Weighted Probabilities ---")
print(f"Unweighted P(A): {p_A:.4f}  |  Weighted P(A): {p_A_weighted:.4f}")
print(f"Unweighted P(B): {p_B:.4f}  |  Weighted P(B): {p_B_weighted:.4f}")

--- Comparison of Unweighted vs Weighted Probabilities ---
Unweighted P(A): 0.1995  |  Weighted P(A): 0.1911
Unweighted P(B): 0.1442  |  Weighted P(B): 0.1408


<div dir="rtl">

**تفسیر:** فراوانی در نمونه لزوماً با برآورد احتمال در جمعیت یکسان نیست. وزن‌های نمونه‌گیری، بیش‌نمونه‌گیری (Oversampling) برخی از گروه‌های خاص را تصحیح می‌کنند تا احتمال واقعی رخداد را در سطح ملی به دست آوریم.

</div>